In [1]:
!pip install -q rdkit pandas
!wget https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_37/chembl_37_chemreps.txt.gz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 46.2 MB/s eta 0:00:00:00:0100:01
--2026-08-24 17:57:21--  https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_37/chembl_37_chemreps.txt.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 292540141 (279M) [application/x-gzip]
Saving to: ‘chembl_37_chemreps.txt.gz’

chembl_37_chemreps. 100%[===================>] 278.99M  44.5MB/s    in 5.5s    

2026-08-24 17:57:27 (50.5 MB/s) - ‘chembl_37_chemreps.txt.gz’ saved [292540141/292540141]



In [2]:
import glob
import gzip
import os
import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog('rdApp.*')


def smiles_to_inchikey(smiles):
  """Generates a standard 27-character InChIKey hash for exact database lookup."""
  if not isinstance(smiles, str) or not smiles.strip():
    return None
  mol = Chem.MolFromSmiles(smiles)
  return Chem.MolToInchiKey(mol) if mol is not None else None


In [3]:
gen_file = '/kaggle/input/datasets/anushkatayal20/generated-molecules/generated_10k_molecules (1).csv'
if not os.path.exists(gen_file) and os.path.exists('generated_molecules.csv'):
  gen_file = 'generated_molecules.csv'

gen_df = pd.read_csv(gen_file)
total_generated = len(gen_df)

# Validate SMILES structures with RDKit
if 'Is_Valid' in gen_df.columns:
  valid_df = gen_df[gen_df['Is_Valid'] == True]
else:
  gen_df['Is_Valid'] = gen_df['Canonical_SMILES'].apply(
      lambda s: Chem.MolFromSmiles(s) is not None
  )
  valid_df = gen_df[gen_df['Is_Valid'] == True]

valid_smiles = valid_df['Canonical_SMILES'].dropna().unique().tolist()
valid_count = len(valid_df)

# Create a set of unique InChIKeys for the generated molecules
gen_inchikeys = {smiles_to_inchikey(s) for s in valid_smiles}
gen_inchikeys.discard(None)
unique_count = len(gen_inchikeys)

In [4]:
zinc_file = '/kaggle/input/datasets/anushkatayal20/zinc-dataset/250k_rndm_zinc_drugs_clean_3.xls'
zinc_novelty_pct = 0.0
len_novel_vs_zinc = 0

if os.path.exists(zinc_file):
  try:
    zinc_df = pd.read_csv(zinc_file)
  except Exception:
    zinc_df = pd.read_excel(zinc_file)

  col = 'smiles' if 'smiles' in zinc_df.columns else zinc_df.columns[0]
  zinc_smiles = zinc_df[col].dropna().unique().tolist()

  # Hash ZINC training structures into InChIKeys
  zinc_inchikeys = {smiles_to_inchikey(s) for s in zinc_smiles}
  zinc_inchikeys.discard(None)

  # Find molecules present in our generated set but missing from ZINC
  novel_vs_zinc = gen_inchikeys - zinc_inchikeys
  len_novel_vs_zinc = len(novel_vs_zinc)
  zinc_novelty_pct = (len_novel_vs_zinc / unique_count) * 100

In [5]:
chembl_files = glob.glob('*chemreps.txt.gz')
chembl_file = chembl_files[0] if chembl_files else None
chembl_novelty_pct = 0.0
len_novel_vs_chembl = 0

if chembl_file and os.path.exists(chembl_file):
  chembl_inchikeys = set()

  with gzip.open(chembl_file, 'rt', encoding='utf-8', errors='ignore') as f:
    next(f, None)
    for line in f:
      parts = line.strip().split('\t')
      if len(parts) >= 4 and parts[3]:
        chembl_inchikeys.add(parts[3])

  novel_vs_chembl = gen_inchikeys - chembl_inchikeys
  len_novel_vs_chembl = len(novel_vs_chembl)
  chembl_novelty_pct = (len_novel_vs_chembl / unique_count) * 100

In [6]:
summary_df = pd.DataFrame({
    'Metric': [
        'Total Generated',
        'Valid Molecules',
        'Validity Rate (%)',
        'Unique Valid Molecules',
        'Uniqueness Rate (%)',
        'Novel vs ZINC 250k',
        'ZINC Novelty Rate (%)',
        'Novel vs ChEMBL 37',
        'ChEMBL Novelty Rate (%)',
    ],
    'Value': [
        f'{total_generated:,}',
        f'{valid_count:,}',
        f'{(valid_count/total_generated)*100:.2f}%',
        f'{unique_count:,}',
        f'{(unique_count/valid_count)*100:.2f}%',
        f'{len_novel_vs_zinc:,}' if os.path.exists(zinc_file) else 'N/A',
        f'{zinc_novelty_pct:.2f}%' if os.path.exists(zinc_file) else 'N/A',
        f'{len_novel_vs_chembl:,}' if chembl_file else 'N/A',
        f'{chembl_novelty_pct:.2f}%' if chembl_file else 'N/A',
    ],
})

summary_df.to_csv('novelty_evaluation_summary.csv', index=False)
summary_df

,Metric,Value
0,Total Generated,"10,000"
1,Valid Molecules,"9,902"
2,Validity Rate (%),99.02%
3,Unique Valid Molecules,"9,877"
4,Uniqueness Rate (%),99.75%
5,Novel vs ZINC 250k,"9,842"
6,ZINC Novelty Rate (%),99.65%
7,Novel vs ChEMBL 37,"9,763"
8,ChEMBL Novelty Rate (%),98.85%


In [7]:
!wget -q https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-InChI-Key.gz

In [8]:
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
retries = Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504]
)
session.mount('https://', HTTPAdapter(max_retries=retries))

url = (
    'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/property/InChIKey/JSON'
)

inchikey_list = list(gen_inchikeys)
batch_size = 200
found_in_pubchem = set()
total_batches = (len(inchikey_list) + batch_size - 1) // batch_size

print(
    f'Checking {len(inchikey_list):,} molecules against PubChem'
    f' ({total_batches} batches)...'
)

for i in range(0, len(inchikey_list), batch_size):
  chunk = inchikey_list[i : i + batch_size]
  payload = {'inchikey': ','.join(chunk)}
  try:
    res = session.post(url, data=payload, timeout=15)
    if res.status_code == 200:
      data = res.json()
      if 'PropertyTable' in data and 'Properties' in data['PropertyTable']:
        for prop in data['PropertyTable']['Properties']:
          if 'InChIKey' in prop:
            found_in_pubchem.add(prop['InChIKey'])
    time.sleep(0.3)  # Gentle rate limit pause
  except Exception as e:
    print(f'    [!] Batch {i//batch_size + 1} retry warning: {e}')

novel_vs_pubchem = gen_inchikeys - found_in_pubchem
len_novel_vs_pubchem = len(novel_vs_pubchem)
pubchem_novelty_pct = (len_novel_vs_pubchem / len(gen_inchikeys)) * 100

print(
    f'Molecules Found in PubChem : {len(found_in_pubchem):,}'
    f' / {len(gen_inchikeys):,}'
)
print(
    f'Novel Molecules vs PubChem : {len_novel_vs_pubchem:,}'
    f' / {len(gen_inchikeys):,}'
)
print(f'PubChem Novelty Rate       : {pubchem_novelty_pct:.2f}%')


Checking 9,877 molecules against PubChem (50 batches)...
Molecules Found in PubChem : 1,783 / 9,877
Novel Molecules vs PubChem : 8,094 / 9,877
PubChem Novelty Rate       : 81.95%


In [9]:
successful_batches = 0
failed_batches = []

for i in range(0, len(inchikey_list), batch_size):
    chunk = inchikey_list[i:i + batch_size]

    try:
        res = session.post(url, data={'inchikey': ','.join(chunk)}, timeout=15)

        if res.status_code == 200:
            successful_batches += 1

            data = res.json()

            if 'PropertyTable' in data and 'Properties' in data['PropertyTable']:
                for prop in data['PropertyTable']['Properties']:
                    if 'InChIKey' in prop:
                        found_in_pubchem.add(prop['InChIKey'])
        else:
            failed_batches.append(i // batch_size + 1)

    except Exception as e:
        failed_batches.append(i // batch_size + 1)

    time.sleep(0.3)

print("Successful batches:", successful_batches)
print("Failed batches:", failed_batches)

Successful batches: 50
Failed batches: []
